## Problem: Add a Benchmark to Your PyTorch Code

### Problem Statement
You are tasked with implementing a simple neural network model with fully connected layers and adding benchmarking functionality to measure and display the time taken for each epoch of training and testing. The goal is to evaluate the model's performance and record the time taken for both training and testing phases.

### Requirements
1. **Define a Neural Network Model**:
   - Implement a simple feedforward neural network using fully connected layers (`nn.Linear`).
   - The network should be suitable for classification tasks.

2. **Benchmark Training and Testing**:
   - Measure the time taken for each epoch during training and display the elapsed time.
   - Measure and display the time taken for the testing phase after each epoch.

### Constraints
- The model should have at least two hidden layers with ReLU activations.
- Use the appropriate loss function and optimizer for training the model.
- Ensure that the benchmarking measures both the training and testing time accurately.

<details>
  <summary>💡 Hint</summary>
  Define the SimpleNN class:
  <br>
  Add two fully connected layers:
  <br>
  Apply a ReLU activation function to the first layer.
  <br>
  <br>
  Benchmark the Code:
  <br>
  Measure and print training time for each epoch.
  <br>
  Measure and print testing time along with accuracy.
</details>

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import time

In [9]:
# Load MNIST dataset
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])

train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)

test_dataset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)

In [10]:
# Define a simple neural network model
# TODO: Add layers to the model
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.module = nn.Sequential(
            # block-1
            nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=16, out_channels=16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            # block-2
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3, padding=1),            
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            # output
            nn.Flatten(),
            nn.Linear(1568, 10),
        )

    def forward(self, x):
        return self.module(x)

# Initialize the model, loss function, and optimizer
model = SimpleNN().to("mps")
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

In [11]:
# Training loop with benchmarking
epochs = 5
for epoch in range(epochs):
    start_time = time.time()  # Start time for training
    for images, labels in train_loader:
        images, labels = images.to("mps"), labels.to("mps")
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    end_time = time.time()  # End time for training
    training_time = end_time - start_time
    print(f"Epoch [{epoch + 1}/{epochs}], Loss: {loss.item():.4f}, Time: {training_time:.4f}s")

Epoch [1/5], Loss: 0.1827, Time: 5.6649s
Epoch [2/5], Loss: 0.1805, Time: 3.7614s
Epoch [3/5], Loss: 0.1821, Time: 3.8112s
Epoch [4/5], Loss: 0.1228, Time: 3.7492s
Epoch [5/5], Loss: 0.0596, Time: 3.8123s


In [12]:
# Evaluate the model on the test set and benchmark the accuracy
correct = 0
total = 0
start_time = time.time()  # Start time for testing
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to("mps"), labels.to("mps")
        
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

end_time = time.time()  # End time for testing
testing_time = end_time - start_time
accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%, Testing Time: {testing_time:.4f}s")

Test Accuracy: 98.06%, Testing Time: 0.8425s
